# 🎙️ Bilingual Speech Emotion Recognition (SER) Demo
### Interactive JupyterLab Notebook for Hindi & English Emotion Recognition

This notebook demonstrates:
- 🔊 Loading and playing English & Hindi emotional speech
- 🎯 Emotion Classification across 7 core emotions
- ⚡ Vocal Emotion Intensity Quantification (0 - 100%)
- 📈 Dynamic Emotion Timeline Tracking (fluctuations over time)
- 💬 Multimodal NLP Transcript Fusion (Devanagari Hindi & English)
- 📊 Interactive Plotly Charts (Radar, Bar, Timeline, Waveform)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import IPython.display as ipd

# Ensure project root is in sys.path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.ser_pipeline import SpeechEmotionRecognizer
from src.utils.visualizer import (
    plot_confidence_distribution,
    plot_emotion_radar,
    plot_emotion_timeline,
    plot_intensity_gauge,
    plot_waveform
)
from src.config import SAMPLE_AUDIO_DIR, EMOTIONS, EMOTION_META

print("✅ Pipeline and visualization modules imported successfully!")

## 1. Load Pre-trained Bilingual Model

In [ ]:
# Initialize recognizer
recognizer = SpeechEmotionRecognizer()
print(f"Model Status: {'Loaded & Ready 🚀' if recognizer.is_loaded else 'Not Loaded ❌'}")
print(f"Emotions Supported: {', '.join(recognizer.emotions)}")

## 2. Test Hindi Audio (खुशी / Happy)
We test with a Hindi sample audio and Devanagari Hindi transcript.

In [ ]:
hi_sample = SAMPLE_AUDIO_DIR / "hindi" / "khushi_happy_sample.wav"
hi_transcript = "आज मैं बहुत खुश हूँ, हमने प्रतियोगिता जीत ली!"

# 1. Audio Player
print("🔊 Hindi Audio Player:")
ipd.display(ipd.Audio(str(hi_sample)))

# 2. Run Inference
hi_results = recognizer.predict_audio(
    audio_input=hi_sample,
    language="hi",
    transcript=hi_transcript,
    audio_weight=0.70
)

# 3. Display Key Metrics
pred_emo = hi_results['predicted_emotion']
inten = hi_results['emotion_intensity']

print("=" * 50)
print(f"🎯 Dominant Emotion:    {EMOTION_META[pred_emo]['emoji']} {pred_emo} ({EMOTION_META[pred_emo]['hi_name']})")
print(f"📊 Confidence Score:     {hi_results['confidence_percentage']}%")
print(f"⚡ Emotion Intensity:    {inten['level_en']} ({inten['level_hi']}) - {inten['intensity_percentage']}%")
print(f"⏱️ Audio Duration:       {hi_results['duration_seconds']}s")
print("=" * 50)

## 3. Interactive Confidence Distribution & Radar Fingerprint

In [ ]:
# Bar Chart
fig_bar = plot_confidence_distribution(hi_results['emotion_probabilities'], hi_results['predicted_emotion'])
fig_bar.show()

# Radar Chart
fig_radar = plot_emotion_radar(hi_results['emotion_probabilities'])
fig_radar.show()

## 4. Dynamic Emotion Timeline (Fluctuations Over Audio Duration)
Tracks how emotion probabilities and vocal intensity fluctuate across sliding windows.

In [ ]:
fig_timeline = plot_emotion_timeline(hi_results['timeline'])
fig_timeline.show()

## 5. Vocal Emotion Intensity Gauge & Acoustic Waveform

In [ ]:
# Intensity Gauge
fig_gauge = plot_intensity_gauge(
    inten['intensity_percentage'],
    inten['level_en'],
    inten['level_hi']
)
fig_gauge.show()

# Waveform
fig_wave = plot_waveform(hi_results['audio_waveform'], hi_results['sample_rate'])
fig_wave.show()

## 6. Test English Audio (Angry)

In [ ]:
en_sample = SAMPLE_AUDIO_DIR / "english" / "angry_sample.wav"
en_transcript = "I am so furious right now, this is completely unacceptable nonsense!"

print("🔊 English Audio Player:")
ipd.display(ipd.Audio(str(en_sample)))

en_results = recognizer.predict_audio(
    audio_input=en_sample,
    language="en",
    transcript=en_transcript,
    audio_weight=0.70
)

print(f"🎯 Dominant Emotion:    {en_results['predicted_emotion']}")
print(f"📊 Confidence Score:     {en_results['confidence_percentage']}%")
print(f"⚡ Emotion Intensity:    {en_results['emotion_intensity']['level_en']} ({en_results['emotion_intensity']['intensity_percentage']}%)")

fig_en_timeline = plot_emotion_timeline(en_results['timeline'])
fig_en_timeline.show()

## 7. Test Any Custom Audio File
Simply pass any path to a `.wav`, `.mp3`, or `.ogg` file below.

In [ ]:
# Example: Test Neutral Hindi audio
test_audio = SAMPLE_AUDIO_DIR / "hindi" / "shant_neutral_sample.wav"
test_transcript = "सब कुछ ठीक है, सामान्य बातचीत चल रही है।"

custom_results = recognizer.predict_audio(test_audio, transcript=test_transcript)

print(f"Result: {custom_results['predicted_emotion']} | Confidence: {custom_results['confidence_percentage']}%")
plot_confidence_distribution(custom_results['emotion_probabilities'], custom_results['predicted_emotion']).show()